# Quantum Convolutional Neural Networks (QCNNs) — Detailed Notes (Session 16)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller  

> **Purpose.** These notes turn the slides into a stand-alone reference for building QCNN-style models: quanvolutional filters, parameter sharing, pooling, and hybrid training with Qiskit ML + PyTorch. We include design patterns, pitfalls, and a minimal code template. Mini-exercises (with brief answers) appear at the end.

---

## Session road-map
1. Recap: why hybrid methods first  
2. What is a QCNN? (quantum analogue of conv+pool)  
3. Quanvolutional filters (patch encoding → PQC → features)  
4. Parameter sharing & translation equivariance  
5. Pooling on quantum circuits (measurement / unitary pooling)  
6. Architectures & training loops (optimisers, losses)  
7. Qiskit ML implementations (EstimatorQNN / SamplerQNN + Torch)  
8. Evaluation vs small classical CNNs  
9. Practical tips, pitfalls, & lab outline

---

## 0) Recap — Hybrid methods → QCNNs
- Hybrid pipelines keep the **quantum piece small** (few qubits, shallow depth) and let the **classical stack** do heavy lifting.  
- QCNNs specialise this idea for **structured data** (images, sequences): apply the **same** small quantum filter across many local patches, just like a classical conv layer.

---

## 1) What is a QCNN?
A QCNN borrows three CNN principles:
1) **Local receptive fields** — operate on small neighbourhoods (e.g., 2×2 patch → 4 qubits).  
2) **Parameter sharing** — the same filter (unitary with parameters $\boldsymbol\theta$) slides over all patches.  
3) **Downsampling / pooling** — reduce dimension by selecting/aggregating qubits or measurements.

**Basic block**
$$
\text{Patch } x_{p}\xrightarrow{\text{encode}} U_\phi(x_{p})\,|0\rangle
\;\xrightarrow{\text{filter }U(\boldsymbol\theta)}\;
|\psi_p\rangle\;\xrightarrow{\text{decode}}\; z_p
$$
where $z_p$ are features (expectations or probabilities) fed to a classical head or next quantum stage.

---

## 2) Quanvolutional filters (quantum “convolutions”)

### Patch → qubits (encoding)
- **Angle encoding (common):** map pixel/sequence values to rotations (e.g., $R_Y(\alpha x)$).  
- **Entangling map:** add ZZ terms to capture local interactions within the patch.  
- **Amplitude encoding:** compact but costly; avoid on NISQ unless patch is tiny and well-normalised.

### Filter circuit $U(\boldsymbol\theta)$
- Hardware-efficient blocks (e.g., RY/RZ + CX) with 1–3 repetitions.  
- Optional **data re-uploading**: interleave $U_\phi$ with trainable layers to boost expressivity without more qubits.

### Decode (measure)
- **Expectations:** $z_{p,i}=\langle Z_{q_i}\rangle$ (stable, few shots).  
- **Probabilities:** full bitstring histogram (richer, more shots).  
- **Fixed projection:** e.g., single “decision” qubit per patch.

**Output shape** is like a feature map: patches × out_channels (number of measured observables).

---

## 3) Parameter sharing (quantum weight tying)
- Reuse the **same** circuit $U(\boldsymbol\theta)$ on every patch; all evaluations share one parameter vector $\boldsymbol\theta$.  
- Implementation detail: **bind** the same `ParameterVector` into all patch circuits or reuse one circuit and **rebind** inputs per patch.

**Benefits:** fewer parameters → better generalisation; encourages **translation equivariance** analogous to classical CNNs.

---

## 4) Pooling for QCNNs

### Measurement pooling (practical)
- Measure a subset of qubits (e.g., keep one “decision” qubit per patch).  
- Or compute simple aggregates (mean / max over $\langle Z\rangle$ from patch qubits) in **classical** post-processing.

### Unitary pooling (conceptual / researchy)
- Apply controlled unitaries between pairs of qubits, then **discard** one qubit (partial trace).  
- Emulates learned pooling but costs more gates and is noise-sensitive.

**Downsampling ratio** ≈ (#kept qubits per patch / #qubits per patch).

---

## 5) QCNN architectures

### Minimal QCNN (images 4×4, patch 2×2)
1. **Quanv layer:** encode 2×2 patch → 4 qubits → filter → output one expectation per patch (stride 2).  
2. **Pooling:** keep that one scalar → 4 outputs total.  
3. **Head:** classical MLP (e.g., 4→16→1) with BCE loss (binary).

### Stacked design
Quanv(2×2) → Pool → Quanv(2×2) on the pooled map → Pool → Dense head.  
Keep depth **shallow** (≤ 2 quanv layers) on NISQ.

---

## 6) Training QCNNs

- **Loss:** BCE (binary), cross-entropy (multi-class), or MSE (regression).  
- **Optimisers:**  
  - *Gradient-based:* Adam with parameter-shift (exact gradients for many native rotations).  
  - *Gradient-free:* SPSA (two evaluations/step → good shot efficiency).  
- **Batching:** process many patches per parameter vector to amortise device latency.  
- **Regularisation:** L2 on $\boldsymbol\theta$; early stopping; small-variance init.

**Barren plateaus:** mitigate with shallow reps, local cost functions (per-patch losses), and feature selection / patch normalisation.

---

## 7) Qiskit ML implementation patterns

> Prefer **primitives** (`Estimator`, `Sampler`) with **EstimatorQNN** / **SamplerQNN** and `TorchConnector`.  
> The older `CircuitQNN`/Opflow stack is legacy.

### A. Expectation-based quanv (EstimatorQNN + PyTorch)
```python
# pip install qiskit qiskit-aer qiskit-machine-learning torch
import torch, numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import TwoLocal
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# 1) Build a 2x2-patch filter on 4 qubits with shared parameters
nq = 4
theta = ParameterVector('θ', length=6)
enc = QuantumCircuit(nq)
# angle-encode patch values later via bound parameters (one RY per qubit)
phi = ParameterVector('x', length=nq)
for q in range(nq):
    enc.ry(phi[q], q)

ans = TwoLocal(nq, rotation_blocks=['ry','rz'], entanglement_blocks='cx',
               entanglement='linear', reps=1, parameter_prefix='θ')
qc = enc.compose(ans)

# 2) QNN that outputs expectation of Z on qubit 0 (one feature per patch)
qnn = EstimatorQNN(circuit=qc,
                   input_params=list(phi),          # per-patch inputs
                   weight_params=list(ans.parameters),  # shared weights
                   estimator=Estimator(shots=1024))

# 3) Torch module wrapper
quanv = TorchConnector(qnn)  # behaves like nn.Module: forward(x_patch) -> scalar

# 4) Slide over image patches (e.g., 4x4 -> four 2x2 patches) outside the QNN:
def image_to_patches(img_4x4):
    # returns shape [num_patches, 4] in row-major 2x2 blocks
    blocks = []
    for r in (0,2):
        for c in (0,2):
            block = img_4x4[r:r+2, c:c+2].reshape(-1)
            blocks.append(block)
    return np.stack(blocks)

# 5) Model: quanv applied per patch + classical head
class QCNNHead(torch.nn.Module):
    def __init__(self, quanv_module):
        super().__init__()
        self.quanv = quanv_module
        self.head  = torch.nn.Sequential(torch.nn.Linear(4, 16),
                                         torch.nn.ReLU(),
                                         torch.nn.Linear(16, 1))
    def forward(self, x_batch_imgs):  # x: [B,4,4]
        feats = []
        for img in x_batch_imgs:
            patches = torch.tensor(image_to_patches(img.numpy()), dtype=torch.float32)
            outs = [self.quanv(p) for p in patches]     # 4 scalars
            feats.append(torch.stack(outs).squeeze())   # shape [4]
        F = torch.stack(feats)                          # [B,4]
        return self.head(F)

model = QCNNHead(quanv)
```

> For speed: vectorise patch evaluation (batch inputs to `EstimatorQNN`), cache transpiled circuits, and standardise pixel ranges (scale to $[-\pi,\pi]$ before binding to `phi`).

### B. Probability-based quanv (SamplerQNN)
Use `SamplerQNN` when you want **bitstring probabilities** for pooling. Convert probabilities to features classically (e.g., sum over selected bitstrings). Stochastic but sometimes richer.

---

## 8) Evaluation & when QCNNs help

### Metrics to report
- **Accuracy / F1** on a fair baseline (logistic regression; 2-layer small CNN).  
- **Parameter count** (QCNN + head vs small CNN).  
- **Shot budget & latency** (circuit evals/epoch).  
- **Robustness** under injected noise (depolarising, readout flips).

### Where QCNNs can shine (today)
- Very small images (e.g., 4×4, 8×8 crops), **few-shot** or class-imbalanced regimes.  
- Data with **strong local correlations** where entanglement within a patch is meaningful.  
- As a **feature extractor** inside a hybrid stack.

---

## 9) Practical tips & pitfalls

- **Encoding range:** normalise/standardise patches; otherwise rotations saturate.  
- **Stride & overlap:** stride 1 inflates compute; start with stride = patch size (no overlap).  
- **Too deep filters:** leads to barren plateaus; 1–2 reps are plenty on NISQ.  
- **Noise:** two-qubit gates dominate errors → keep entanglement minimal and topology-aware.  
- **Batching:** evaluate many patches per $\boldsymbol\theta$ to amortise primitive latency.  
- **Readout calibration:** invert confusion matrix to debias $\langle Z\rangle$.  
- **Deprecations:** prefer `EstimatorQNN`/`SamplerQNN` over legacy `CircuitQNN` (Opflow).

---

## 10) Worked example (conceptual pipeline)
1) **Dataset:** MNIST 4×4 downsample (or 8×8 cropped digits).  
2) **Preprocess:** scale pixels to $[0,1]$ → map to angles via $\pi x$.  
3) **Quanv:** 2×2 patch → 4-qubit filter with RY/RZ+CX (shared $\boldsymbol\theta$); output one $\langle Z_0\rangle$ per patch.  
4) **Pooling:** keep one feature per patch → 4 (or 16) features per image.  
5) **Head:** 2-layer MLP; BCE loss.  
6) **Optimiser:** SPSA or Adam (parameter-shift); shots start at 512, increase to 2048.  
7) **Report:** accuracy vs shots; ablation: classical conv of same receptive field & parameter count.

---

## 11) Mini-exercises (answers in Appendix)

1. **Receptive field.** For two stacked quanv layers with 2×2 patches and stride 2, what is the effective receptive field size on the input image?  
2. **Parameter sharing.** Show how to bind the *same* `ParameterVector('θ', k)` to all patch circuits in Qiskit without duplicating parameters.  
3. **Pooling trade-off.** Compare measuring (i) one qubit per patch vs (ii) full probability vector. How do shots and feature richness scale?  
4. **Noise budget.** With two-qubit error 1% and 6 entangling gates per patch, estimate the multiplicative shrink in expectation values and propose two mitigations.  
5. **Baseline parity.** Design a tiny classical CNN with roughly the same parameter count as your QCNN head; specify conv kernel size, channels, and head.

---

## 12) Summary (Session 16)
- QCNNs apply **shared**, **local** PQCs to patches (quanvolution) plus **pooling** to reduce dimension—mirroring CNNs.  
- Keep filters **small and shallow**, bind shared parameters, and batch patches to handle device latency.  
- Use `EstimatorQNN` (expectations) or `SamplerQNN` (probabilities) with `TorchConnector` for practical training.  
- Evaluate against **matched** classical baselines and report shot/latency costs.  
- QCNNs are promising for **tiny, structured datasets** and as **hybrid feature extractors** on NISQ hardware.

---

## 13) Looking ahead
- **Next Session:** Quantum Graph Neural Networks (QGNNs) — message-passing analogues with entangling operations over graph edges.  
- **Homework 4 (QCNN):**  
  1) Implement a 2×2 quanv layer on 4×4 digits; report accuracy vs shots (256→4096).  
  2) Compare expectation-based vs probability-based decoding under the same shot budget.  
  3) Benchmark against a 1-layer classical CNN with similar parameter count.

---

## Appendix — mini-exercise solutions (sketch)

1. **Receptive field:** Layer1 covers 2×2; after pooling/stride-2, Layer2 sees 2×2 **of Layer1 patches** → overall 4×4 on the input.  
2. **Shared params:** Create one circuit with `ParameterVector('θ', k)`. Reuse this circuit and **bind different `phi` (data) values per patch**; the `θ` symbols are the same object across all binds, so they share values.  
3. **Pooling trade-off:** (i) One-qubit expectation → 1 scalar/patch, low variance (needs O(1/ε²) shots). (ii) Full histogram → $2^{q}$ values (for q qubits), richer but shot cost grows with $2^{q}$; aggregation needed to avoid high variance.  
4. **Shrink:** Roughly $(1-0.01)^{6} \approx 0.94$. Mitigate with (a) topology-aware routing to reduce entangling gates/SWAPs, (b) zero-noise extrapolation + dynamical decoupling; also consider fewer entanglers.  
5. **Baseline CNN:** e.g., Conv2d(1→1, kernel=2×2, stride=2, no bias) → 1 feature per patch (4 params). Head: Linear(4→16→1) ~ 4×16+16×1 ≈ 80 params. Compare to QCNN head params of similar magnitude.
